In [12]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from multiprocessing import cpu_count

from model.proposed import MYModule

### 可复现性

In [13]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


### hyperparameter tuning

In [14]:
SEQ_LEN = 92
BATCH_SIZE = 128
EMBED_DIM = 128
DROP_OUT = 0.8
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()
# N_GCN_LAYERS = 5
Q_is_S = True
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


### load assist data

In [15]:
# dataset_name = 'assist12'
# dataset_path = os.path.join(os.getcwd(), 'dataset', dataset_name)

# df = pd.read_csv(os.path.join(dataset_path, "assist.csv"), low_memory=False, encoding="ISO-8859-1")
# df.columns

In [16]:
# train : validation = 80% : 20%
df_train = pd.read_csv('dataset/assist12/train.csv', low_memory=False, encoding="ISO-8859-1")
df_val = pd.read_csv('dataset/assist12/test.csv', low_memory=False, encoding="ISO-8859-1")
df_train.columns

Index(['user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response',
       'attempt_count', 'correct'],
      dtype='object')

In [17]:
key_user = 'user_id'
key_q = 'q_idx'
key_s = 's_idx'
key_qtype = 'q_type'
key_diff = 'q_diff'
key_ms = 'ms_first_response'
key_attempt = 'attempt_count'
key_correct = 'correct'

N_QUESTION = 50988
N_SKILL = 198
N_QUESTION_TYPE = 6

print("num of question:{}, num of skill:{}, n_question_type:{}".format(N_QUESTION, N_SKILL, N_QUESTION_TYPE))


num of question:50988, num of skill:198, n_question_type:6


In [18]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
def generate_group_by_df(df):
    KEY = key_s if Q_is_S else key_q
    group = df.groupby([key_user]).apply(lambda r: (
                r[KEY].values,
                r[key_s].values,
                r[key_qtype].values,
                r[key_diff].values,
                r[key_ms].values,
                r[key_attempt].values,
                r[key_correct].values                                                                                                                                                                                                                                      
                ))
    return group



train = generate_group_by_df(df_train) 
val = generate_group_by_df(df_val)
train.head()

len_list = [len(train.iloc[i][0]) for i in range(len(train))]

N_QUERY_FEATURES = len(train.iloc[0])-1
print("N_QUERY_FEATURES:{}".format(N_QUERY_FEATURES))
print("seq_len mean:{}, max:{}, min:{}".format(np.mean(len_list), np.max(len_list), np.min(len_list)))

N_QUERY_FEATURES:6
seq_len mean:91.76676637620844, max:2045, min:1


###  assist12 dataset

In [19]:
from data_loader.assist09 import Assist09Dataset
from data_loader.saintdataset import SAINTDataset

N_Q_OR_S = N_SKILL if Q_is_S else N_QUESTION


train_dataset = SAINTDataset(train, N_Q_OR_S, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = SAINTDataset(val, N_Q_OR_S, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))

train:23067, test:5767


In [20]:
# train_dataset[0][2]

In [21]:
import warnings
warnings.filterwarnings('ignore')

from model.saint2 import SAINTModule


model = SAINTModule(
    dim_model=EMBED_DIM,
    num_en=6,
    num_de=6,
    heads_en=8,
    heads_de=8,
    total_ex=N_Q_OR_S,
    total_cat=N_QUESTION_TYPE,
    total_in=2,
    seq_len=SEQ_LEN
)
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

patience = 6 if N_QUERY_FEATURES == 6 else 3
# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)
print("patience:{}".format(patience))


GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


patience:6


In [22]:
trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type              | Params
--------------------------------------------
0 | loss  | BCEWithLogitsLoss | 0     
1 | model | saint             | 1.8 M 
--------------------------------------------
1.8 M     Trainable params
0         Non-trainable params
1.8 M     Total params
7.005     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 0, global step 181: 'v_auc' reached 0.50792 (best 0.50792), saving model to '/home/czy/KT/BRIKT/lightning_logs/version_97/checkpoints/epoch=0-step=181.ckpt' as top 1


...

Epoch 17, global step 3258: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 18, global step 3439: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 19, global step 3620: 'v_auc' was not in top 1
